# FlashRank: Ultra-Light Reranking for RAG

### by Mayank Chugh | Advanced RAG Series (2026 Edition) | Episode 6

Reranking is the second stage of two-stage retrieval: a fast first-stage search returns a shortlist, and a more careful model re-scores that shortlist before anything reaches the LLM. This notebook uses **FlashRank**, an ONNX-based reranking library that runs on CPU, first on its own and then inside a LangChain RAG pipeline over the 2022 State of the Union address.

| Component | Choice |
|---|---|
| Reranker | FlashRank 0.2.10 (ONNX Runtime, CPU): Nano, Small, Medium, Large |
| Embeddings | BAAI/bge-small-en-v1.5 (CPU, normalized) |
| Vector store | FAISS (faiss-cpu, in-memory) |
| Reranking in LangChain | `FlashrankRerank` + `ContextualCompressionRetriever` |
| LLM | Qwen/Qwen3-4B-Instruct-2507 (local, 4-bit) → claude-sonnet-5 → gpt-5.6-sol via `with_fallbacks()` |
| Corpus | `state_of_the_union.txt`, 96 chunks (500 characters, 100 overlap) |

**Runtime:** T4 GPU. Only the local LLM needs the GPU; embeddings, FAISS and FlashRank all run on CPU.

**Colab Secrets needed for the LLM section:** `ANTHROPIC_API_KEY`, `OPENAI_API_KEY` (used only if a model earlier in the fallback chain raises an error).

> 📊 **Reference run** (CPU sandbox, 1 vCPU, ONNX Runtime 1.24.4, 15 Sept 2026). Colab timings will differ; rankings and scores are the numbers to compare.
>
> **Part 1, warm reranking of 5 passages:** Nano 11.04 ms · Small 247.79 ms · Medium 461.32 ms · Large 433.66 ms
>
> **Part 2, FAISS top-10 chunk ids:** 73, 72, 88, 79, 86, 2, 19, 0, 95, 18
>
> **`FlashrankRerank()` default model:** `ms-marco-MultiBERT-L-12` → top-3 73 (0.9988) · 2 (0.9941) · 19 (0.8627)
>
> **Pinned `ms-marco-MiniLM-L-12-v2`:** top-3 73 (0.99303) · 2 (0.00005) · 79 (0.00002) · with `score_threshold=0.5`: [73]

> ⚠️ **Pending the live T4 run:** the local Qwen3 answer text. The chain wiring and fallback behaviour were verified with stand-in models; no answer text is quoted anywhere until the notebook runs on Colab.


## FlashRank's Four Model Options

| Option | Model name | README size | ONNX file on disk (measured) | Notes |
|---|---|---|---|---|
| Nano | `ms-marco-TinyBERT-L-2-v2` | ~4 MB | 4.5 MB | FlashRank's default |
| Small | `ms-marco-MiniLM-L-12-v2` | ~34 MB | 34.0 MB | README: best cross-encoder reranker |
| Medium | `rank-T5-flan` | ~110 MB | 110.7 MB | README: best non cross-encoder reranker |
| Large | `ms-marco-MultiBERT-L-12` | ~150 MB | 168.2 MB | Multilingual, 100+ languages. README: don't use for English |

FlashRank also ships `ce-esci-MiniLM-L12-v2`, `miniReranker_arabic_v1`, and a listwise LLM reranker (`rank_zephyr_7b_v1_full`, ~4 GB GGUF, needs `pip install flashrank[listwise]`). This notebook covers the four pairwise options above.


## 1. Install FlashRank

In [1]:
!pip install -q flashrank==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.4 MB/s eta 0:00:00


In [14]:
import importlib.metadata as metadata

for package in ["flashrank", "onnxruntime", "tokenizers", "numpy"]:
    print(f"{package:<12} {metadata.version(package)}")

flashrank    0.2.10
onnxruntime  1.31.0
tokenizers   0.23.2
numpy        2.1.3


## 2. The Query and Candidate Passages

In [15]:
query = "How to speedup LLMs?"

In [16]:
passages = [
   {
      "id":1,
      "text":"Introduce *lookahead decoding*: - a parallel decoding algo to accelerate LLM inference - w/o the need for a draft model or a data store - linearly decreases # decoding steps relative to log(FLOPs) used per decoding step.",
      "meta": {"additional": "info1"}
   },
   {
      "id":2,
      "text":"LLM inference efficiency will be one of the most crucial topics for both industry and academia, simply because the more efficient you are, the more $$$ you will save. vllm project is a must-read for this direction, and now they have just released the paper",
      "meta": {"additional": "info2"}
   },
   {
      "id":3,
      "text":"There are many ways to increase LLM inference throughput (tokens/second) and decrease memory footprint, sometimes at the same time. Here are a few methods I’ve found effective when working with Llama 2. These methods are all well-integrated with Hugging Face. This list is far from exhaustive; some of these techniques can be used in combination with each other and there are plenty of others to try. - Bettertransformer (Optimum Library): Simply call `model.to_bettertransformer()` on your Hugging Face model for a modest improvement in tokens per second. - Fp4 Mixed-Precision (Bitsandbytes): Requires minimal configuration and dramatically reduces the model's memory footprint. - AutoGPTQ: Time-consuming but leads to a much smaller model and faster inference. The quantization is a one-time cost that pays off in the long run.",
      "meta": {"additional": "info3"}
   },
   {
      "id":4,
      "text":"Ever want to make your LLM inference go brrrrr but got stuck at implementing speculative decoding and finding the suitable draft model? No more pain! Thrilled to unveil Medusa, a simple framework that removes the annoying draft model while getting 2x speedup.",
      "meta": {"additional": "info4"}
   },
   {
      "id":5,
      "text":"vLLM is a fast and easy-to-use library for LLM inference and serving. vLLM is fast with: State-of-the-art serving throughput Efficient management of attention key and value memory with PagedAttention Continuous batching of incoming requests Optimized CUDA kernels",
      "meta": {"additional": "info5"}
   }
]

## 3. Import FlashRank and Name the Four Options

In [17]:
from flashrank import Ranker, RerankRequest

In [19]:
CACHE_DIR = "/content/flashrank_models"

MODEL_OPTIONS = {
    "Nano": "ms-marco-TinyBERT-L-2-v2",
    "Small": "ms-marco-MiniLM-L-12-v2",
    "Medium": "rank-T5-flan",
    "Large": "ms-marco-MultiBERT-L-12",
}

In [20]:
def get_result(query, passages, choice):
    ranker = Ranker(model_name=MODEL_OPTIONS[choice], cache_dir=CACHE_DIR)
    request = RerankRequest(query=query, passages=passages)
    results = ranker.rerank(request)
    for r in results:
        print(f"id={r['id']}  score={float(r['score']):.6f}")
    return results

## 4. Rerank With Each Model (Timed)

In [21]:
%%time
print("timing baseline")

timing baseline
CPU times: user 63 µs, sys: 11 µs, total: 74 µs
Wall time: 78.2 µs


In [22]:
%%time
nano_results = get_result(query, passages, "Nano")

id=4  score=0.018154
id=5  score=0.013988
id=3  score=0.000919
id=1  score=0.000762
id=2  score=0.000285
CPU times: user 107 ms, sys: 2.8 ms, total: 110 ms
Wall time: 115 ms


In [23]:
%%time
small_results = get_result(query, passages, "Small")

id=4  score=0.943004
id=3  score=0.847676
id=1  score=0.803419
id=5  score=0.557645
id=2  score=0.004488
CPU times: user 692 ms, sys: 19.3 ms, total: 711 ms
Wall time: 710 ms


In [24]:
%%time
medium_results = get_result(query, passages, "Medium")

id=3  score=0.511264
id=4  score=0.504936
id=2  score=0.495559
id=1  score=0.477897
id=5  score=0.426637
CPU times: user 1.77 s, sys: 83.7 ms, total: 1.86 s
Wall time: 1.85 s


In [26]:
%%time
large_results = get_result(query, passages, "Large")

id=3  score=0.997602
id=4  score=0.997349
id=5  score=0.991014
id=1  score=0.977863
id=2  score=0.773594
CPU times: user 1.99 s, sys: 188 ms, total: 2.18 s
Wall time: 2.2 s


In [27]:
!du -sh /content/flashrank_models/*

124K	/content/flashrank_models/__MACOSX
33M	/content/flashrank_models/ms-marco-MiniLM-L-12-v2
162M	/content/flashrank_models/ms-marco-MultiBERT-L-12
4.8M	/content/flashrank_models/ms-marco-TinyBERT-L-2-v2
108M	/content/flashrank_models/rank-T5-flan


## 5. What %%time Actually Measured

In [28]:
import copy
import time
import statistics

def benchmark(choice, runs=30):
    t0 = time.perf_counter()
    ranker = Ranker(model_name=MODEL_OPTIONS[choice], cache_dir=CACHE_DIR, log_level="WARNING")
    load_ms = (time.perf_counter() - t0) * 1000
    for _ in range(3):
        ranker.rerank(RerankRequest(query=query, passages=copy.deepcopy(passages)))
    timings = []
    for _ in range(runs):
        batch = copy.deepcopy(passages)
        t0 = time.perf_counter()
        ranker.rerank(RerankRequest(query=query, passages=batch))
        timings.append((time.perf_counter() - t0) * 1000)
    return round(load_ms, 1), round(statistics.median(timings), 2)

In [29]:
for choice in MODEL_OPTIONS:
    load_ms, rerank_ms = benchmark(choice)
    print(f"{choice:<7} load: {load_ms:>7} ms   rerank: {rerank_ms:>8} ms")

Nano    load:    56.1 ms   rerank:    15.33 ms
Small   load:   193.5 ms   rerank:   488.46 ms
Medium  load:   409.7 ms   rerank:  1472.24 ms
Large   load:   701.3 ms   rerank:  1360.31 ms


## 6. Gotcha: `rerank()` Changes the List You Pass In

In [30]:
[p["id"] for p in passages]

[3, 4, 5, 1, 2]

In [31]:
passages[0]

{'id': 3,
 'text': "There are many ways to increase LLM inference throughput (tokens/second) and decrease memory footprint, sometimes at the same time. Here are a few methods I’ve found effective when working with Llama 2. These methods are all well-integrated with Hugging Face. This list is far from exhaustive; some of these techniques can be used in combination with each other and there are plenty of others to try. - Bettertransformer (Optimum Library): Simply call `model.to_bettertransformer()` on your Hugging Face model for a modest improvement in tokens per second. - Fp4 Mixed-Precision (Bitsandbytes): Requires minimal configuration and dramatically reduces the model's memory footprint. - AutoGPTQ: Time-consuming but leads to a much smaller model and faster inference. The quantization is a one-time cost that pays off in the long run.",
 'meta': {'additional': 'info3'},
 'score': np.float32(0.9976015)}

In [32]:
import json

try:
    json.dumps(passages)
except TypeError as err:
    print("TypeError:", err)

TypeError: Object of type float32 is not JSON serializable


In [33]:
def to_json_safe(results):
    return [{**r, "score": float(r["score"])} for r in results]

print(json.dumps(to_json_safe(passages)[:1], indent=2))

[
  {
    "id": 3,
    "text": "There are many ways to increase LLM inference throughput (tokens/second) and decrease memory footprint, sometimes at the same time. Here are a few methods I\u2019ve found effective when working with Llama 2. These methods are all well-integrated with Hugging Face. This list is far from exhaustive; some of these techniques can be used in combination with each other and there are plenty of others to try. - Bettertransformer (Optimum Library): Simply call `model.to_bettertransformer()` on your Hugging Face model for a modest improvement in tokens per second. - Fp4 Mixed-Precision (Bitsandbytes): Requires minimal configuration and dramatically reduces the model's memory footprint. - AutoGPTQ: Time-consuming but leads to a much smaller model and faster inference. The quantization is a one-time cost that pays off in the long run.",
    "meta": {
      "additional": "info3"
    },
    "score": 0.9976015090942383
  }
]


## 7. Do the Models Agree?

In [34]:
for choice, model_name in MODEL_OPTIONS.items():
    ranker = Ranker(model_name=model_name, cache_dir=CACHE_DIR, log_level="WARNING")
    ranked = ranker.rerank(RerankRequest(query=query, passages=copy.deepcopy(passages)))
    print(f"{choice:<7}", [r["id"] for r in ranked], [round(float(r["score"]), 4) for r in ranked])

Nano    [4, 5, 3, 1, 2] [0.0182, 0.014, 0.0009, 0.0008, 0.0003]
Small   [4, 3, 1, 5, 2] [0.943, 0.8477, 0.8034, 0.5576, 0.0045]
Medium  [3, 4, 2, 1, 5] [0.5113, 0.5049, 0.4956, 0.4779, 0.4266]
Large   [3, 4, 5, 1, 2] [0.9976, 0.9973, 0.991, 0.9779, 0.7736]


# Part 2 — FlashRank Inside a LangChain RAG Pipeline

## 8. Install the RAG Stack

In [35]:
!pip install -q langchain langchain-classic langchain-community langchain-text-splitters \
    langchain-huggingface langchain-anthropic langchain-openai faiss-cpu sentence-transformers
!pip install -q bitsandbytes accelerate "transformers>=4.51.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.1/146.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━

In [36]:
for package in ["langchain", "langchain-classic", "langchain-community", "langchain-huggingface",
                "sentence-transformers", "faiss-cpu", "transformers"]:
    print(f"{package:<22} {metadata.version(package)}")

langchain              1.4.3
langchain-classic      1.0.8
langchain-community    0.4.2
langchain-huggingface  1.2.3
sentence-transformers  5.7.0
faiss-cpu              1.15.1
transformers           5.18.0


## 9. Load and Chunk the Corpus

In [37]:
import logging

for noisy_logger in ["httpx", "faiss.loader", "sentence_transformers"]:
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

In [38]:
!wget -q -O /content/state_of_the_union.txt https://raw.githubusercontent.com/langchain-ai/langchain/v0.3/docs/docs/how_to/state_of_the_union.txt

In [39]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_512/2253663707.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [40]:
documents = TextLoader("/content/state_of_the_union.txt").load()

In [41]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [42]:
texts = text_splitter.split_documents(documents)

In [43]:
for chunk_id, text in enumerate(texts):
    text.metadata["id"] = chunk_id

In [44]:
len(texts), max(len(t.page_content) for t in texts)

(96, 496)

In [45]:
texts[0]

Document(metadata={'source': '/content/state_of_the_union.txt', 'id': 0}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny.')

## 10. Ground Truth: Which Chunk Answers the Question?

In [46]:
[t.metadata["id"] for t in texts if "Ketanji" in t.page_content]

[73]

## 11. Embeddings and FAISS (CPU)

In [47]:
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [48]:
%%time
vectorstore = FAISS.from_documents(texts, embedding)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

CPU times: user 5.89 s, sys: 76.4 ms, total: 5.96 s
Wall time: 6.18 s


## 12. First-Stage Retrieval: Top 10

In [49]:
query = "What did the president say about Ketanji Brown Jackson"

In [50]:
docs = retriever.invoke(query)
print([doc.metadata["id"] for doc in docs])

[73, 72, 88, 79, 86, 2, 19, 0, 95, 18]


In [51]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [
                f"Document {i+1}:\n\n{d.page_content}\nMetadata: {d.metadata}"
                for i, d in enumerate(docs)
            ]
        )
    )

In [52]:
pretty_print_docs(docs)

Document 1:

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.
Metadata: {'source': '/content/state_of_the_union.txt', 'id': 73}
----------------------------------------------------------------------------------------------------
Document 2:

We cannot let this happen. 

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your s

## 13. Second Stage: `FlashrankRerank` With Its Defaults

In [53]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

In [54]:
compressor = FlashrankRerank()
compressor.model, compressor.top_n, compressor.score_threshold

ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:01<00:00, 70.0MiB/s]


('ms-marco-MultiBERT-L-12', 3, 0.0)

In [55]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)
compressed_docs = compression_retriever.invoke(query)
len(compressed_docs)

3

In [56]:
for doc in compressed_docs:
    print(doc.metadata["id"], round(float(doc.metadata["relevance_score"]), 4))

73 0.9988
2 0.9977
19 0.9753


In [57]:
pretty_print_docs(compressed_docs)

Document 1:

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.
Metadata: {'id': 73, 'relevance_score': np.float32(0.99881303), 'source': '/content/state_of_the_union.txt'}
----------------------------------------------------------------------------------------------------
Document 2:

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world. 

Groups of citizens blocking tanks with their bodies. Everyone from students to retirees teachers turned soldiers defending their homeland. 

In this struggle as President Zelenskyy said in his speech to the European Parliament “Light will win over darkness.” The Ukrainian Amba

## 14. Same Ten Chunks, Four Rerankers

In [58]:
def compare_rerankers(docs, query, relevant_id=73, runs=10):
    for choice, model_name in MODEL_OPTIONS.items():
        reranker = FlashrankRerank(
            client=Ranker(model_name=model_name, cache_dir=CACHE_DIR, log_level="WARNING"),
            model=model_name,
            top_n=len(docs),
        )
        reranker.compress_documents(docs, query)
        timings = []
        for _ in range(runs):
            t0 = time.perf_counter()
            ranked = reranker.compress_documents(docs, query)
            timings.append((time.perf_counter() - t0) * 1000)
        order = [d.metadata["id"] for d in ranked]
        top3 = "  ".join(f"{d.metadata['id']}:{float(d.metadata['relevance_score']):.5f}" for d in ranked[:3])
        print(f"{choice:<7} {statistics.median(timings):>8.2f} ms | top-3 {top3} | chunk {relevant_id} at rank {order.index(relevant_id) + 1}")

In [59]:
compare_rerankers(docs, query)

Nano       18.55 ms | top-3 73:0.99402  2:0.00003  79:0.00003 | chunk 73 at rank 1
Small     515.16 ms | top-3 73:0.99387  2:0.00005  79:0.00002 | chunk 73 at rank 1
Medium   1611.83 ms | top-3 0:0.45517  72:0.43817  79:0.41294 | chunk 73 at rank 6
Large    1564.36 ms | top-3 73:0.99881  2:0.99775  19:0.97531 | chunk 73 at rank 1


## 15. Pin an English Model and Add a Score Threshold

In [60]:
english_reranker = FlashrankRerank(
    client=Ranker(model_name="ms-marco-MiniLM-L-12-v2", cache_dir=CACHE_DIR, log_level="WARNING"),
    model="ms-marco-MiniLM-L-12-v2",
    top_n=3,
    score_threshold=0.5,
)
default_with_threshold = FlashrankRerank(top_n=3, score_threshold=0.5)

print("MiniLM-L-12 + threshold 0.5:", [d.metadata["id"] for d in english_reranker.compress_documents(docs, query)])
print("Default     + threshold 0.5:", [d.metadata["id"] for d in default_with_threshold.compress_documents(docs, query)])

MiniLM-L-12 + threshold 0.5: [73]
Default     + threshold 0.5: [73, 2, 19]


In [61]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=english_reranker, base_retriever=retriever
)
compressed_docs = compression_retriever.invoke(query)
pretty_print_docs(compressed_docs)

Document 1:

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.
Metadata: {'id': 73, 'relevance_score': np.float32(0.99386615), 'source': '/content/state_of_the_union.txt'}


In [62]:
def context_chars(documents):
    return sum(len(d.page_content) for d in documents)

print("FAISS top-10:       ", context_chars(docs))
print("Default top-3:      ", context_chars(compressor.compress_documents(docs, query)))
print("MiniLM + threshold: ", context_chars(compressed_docs))

FAISS top-10:        4026
Default top-3:       1245
MiniLM + threshold:  332


## 16. Robustness Check: Swapped Name Order

In [63]:
swapped_query = "What did the president say about Ketanji Jackson Brown"

print("FAISS top-10:        ", [d.metadata["id"] for d in retriever.invoke(swapped_query)])
print("After MiniLM rerank: ", [d.metadata["id"] for d in compression_retriever.invoke(swapped_query)])

FAISS top-10:         [73, 72, 79, 88, 86, 2, 0, 14, 90, 19]
After MiniLM rerank:  [73]


## 17. LLM: Local Qwen3 First, Paid APIs as Fallbacks

In [64]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [65]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

model_id = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [66]:
local_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,
)
primary_llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=local_pipeline))

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [67]:
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI

fallback_llm_1 = ChatAnthropic(model="claude-sonnet-5")
fallback_llm_2 = ChatOpenAI(model="gpt-5.6-sol")

llm = primary_llm.with_fallbacks([fallback_llm_1, fallback_llm_2])

## 18. The RAG Chain

In [68]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below. "
    "If the context does not contain the answer, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def format_docs(retrieved):
    return "\n\n".join(doc.page_content for doc in retrieved)

rag_chain = (
    {
        "context": itemgetter("question") | compression_retriever | format_docs,
        "question": itemgetter("question"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [69]:
answer = rag_chain.invoke({"question": query})
print(answer)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The president said that they nominated Ketanji Brown Jackson, describing her as one of the nation’s top legal minds and someone who will continue Justice Breyer’s legacy of excellence.


## Key Takeaways

1. Measure reranking latency with a loaded model, not with `%%time` around a function that builds the model.
2. `FlashrankRerank()` with no arguments loads `ms-marco-MultiBERT-L-12`, the multilingual model FlashRank's README says not to use for English. Pin the model.
3. Scores are not comparable across models, so a `score_threshold` has to be tuned per model.
4. `rerank()` sorts the list you pass in and returns `numpy.float32` scores. Pass a copy, convert before JSON.
5. Check rankings against a known answer: on this query the Medium model placed the only relevant chunk seventh.
